<a href="https://colab.research.google.com/github/gabrielcabanilla1/EXSU-500-Clinical-NLP-large-language-models-rct/blob/main/notebooks/02_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Reads PubMed 20k RCT, normalises and verifies the data, adds the columns used by all models,
and saves the prepared files to Google Drive. Shared by the whole team.

**Normalisation choices.** We only apply light normalisation (standard Unicode form and whitespace
cleaning). We do not remove stop words or apply stemming, because function words such as "to" or
"we" are strong cues for the rhetorical role of a sentence. Lower-casing is done later by each
model: by the uncased PubMedBERT tokenizer and by the TF-IDF vectorizer.

In [1]:
# ===== CONFIGURATION =====
DATA_VERSION = 'PubMed_20k_RCT'

DATA_DIR = f'pubmed-rct/{DATA_VERSION}'
DRIVE_DIR = '/content/drive/MyDrive/exsu_project/prepared_data'
CLASSES = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']
N_POSITION_BINS = 10

EXPECTED = {'train': (15000, 180040), 'dev': (2500, 30212), 'test': (2500, 30135)}

In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists('pubmed-rct'):
    !git clone --depth 1 https://github.com/Franck-Dernoncourt/pubmed-rct.git
print('Data folder found:', os.path.exists(DATA_DIR))

Mounted at /content/drive
Cloning into 'pubmed-rct'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 19 (delta 5), reused 14 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 177.08 MiB | 12.64 MiB/s, done.
Resolving deltas: 100% (5/5), done.
Updating files: 100% (13/13), done.
Data folder found: True


In [3]:
import pandas as pd
import numpy as np

def read_split(path):
    """Read one split file. Returns one row per sentence."""
    rows, sentences, abstract_id = [], [], None

    def save_abstract():
        n = len(sentences)
        for i, (label, text) in enumerate(sentences):
            rows.append({'abstract_id': abstract_id, 'sentence_idx': i, 'n_sentences': n,
                         'label': label, 'text': text})

    for line in open(path, encoding='utf-8'):
        line = line.rstrip('\n')
        if line.startswith('###'):
            abstract_id, sentences = line[3:], []
        elif line.strip() == '':
            save_abstract()
            sentences = []
        else:
            label, text = line.split('\t', 1)
            sentences.append((label, text))
    save_abstract()
    return pd.DataFrame(rows)

data = {split: read_split(f'{DATA_DIR}/{split}.txt') for split in ['train', 'dev', 'test']}
for split, df in data.items():
    print(f'{split:5s}: {df.abstract_id.nunique():6,d} abstracts | {len(df):7,d} sentences')

train: 15,000 abstracts | 180,040 sentences
dev  :  2,500 abstracts |  30,212 sentences
test :  2,500 abstracts |  30,135 sentences


In [4]:
import unicodedata

def normalise(text):
    text = unicodedata.normalize('NFC', text)
    return ' '.join(text.split())

for split, df in data.items():
    before = df.text.copy()
    df['text'] = df.text.map(normalise)
    print(f'{split:5s}: {(before != df.text).sum():,} sentences changed by whitespace/Unicode normalisation')

train: 0 sentences changed by whitespace/Unicode normalisation
dev  : 0 sentences changed by whitespace/Unicode normalisation
test : 0 sentences changed by whitespace/Unicode normalisation


In [5]:
problems = []

for split, df in data.items():
    n_abstracts, n_sentences = df.abstract_id.nunique(), len(df)
    if (n_abstracts, n_sentences) != EXPECTED[split]:
        problems.append(f'{split}: {n_abstracts} abstracts / {n_sentences} sentences, expected {EXPECTED[split]}')
    unknown_labels = set(df.label) - set(CLASSES)
    if unknown_labels:
        problems.append(f'{split}: unknown labels {unknown_labels}')
    n_empty = (df.text.str.len() == 0).sum()
    if n_empty:
        problems.append(f'{split}: {n_empty} empty sentences')
    if df.duplicated(['abstract_id', 'sentence_idx']).any():
        problems.append(f'{split}: repeated (abstract, sentence) pairs')

for a, b in [('train', 'dev'), ('train', 'test'), ('dev', 'test')]:
    shared = set(data[a].abstract_id) & set(data[b].abstract_id)
    if shared:
        problems.append(f'{len(shared)} abstracts shared between {a} and {b}')

if problems:
    print('⚠️ PROBLEMS FOUND:')
    for p in problems:
        print('  -', p)
else:
    print('✅ All checks passed: expected sizes, valid labels, no empty sentences, no abstract in two splits.')

train_sentences = set(data['train'].text)
for split in ['dev', 'test']:
    pct = data[split].text.isin(train_sentences).mean() * 100
    print(f'{split}: {pct:.1f}% of sentences also appear word for word in train')

✅ All checks passed: expected sizes, valid labels, no empty sentences, no abstract in two splits.
dev: 0.9% of sentences also appear word for word in train
test: 1.0% of sentences also appear word for word in train


In [6]:
def add_columns(df):
    df = df.copy()
    df['y'] = df.label.map(CLASSES.index)                                       # label as a number 0..4
    df['position'] = df.sentence_idx / (df.n_sentences - 1).clip(lower=1)      # 0 = first, 1 = last
    df['position_bin'] = np.minimum((df.position * N_POSITION_BINS).astype(int), N_POSITION_BINS - 1)
    df['n_words'] = df.text.str.split().str.len()

    by_abstract = df.groupby('abstract_id', sort=False).text
    df['previous'] = by_abstract.shift(1).fillna('none')    # previous sentence of the SAME abstract
    df['next'] = by_abstract.shift(-1).fillna('none')       # next sentence of the SAME abstract
    df['context'] = ('sentence ' + (df.sentence_idx + 1).astype(str) + ' of ' + df.n_sentences.astype(str)
                     + ' . previous: ' + df['previous'] + ' next: ' + df['next'])
    return df

data = {split: add_columns(df) for split, df in data.items()}
data['dev'].head(3)

,abstract_id,sentence_idx,n_sentences,label,text,y,position,position_bin,n_words,previous,next,context
0,24290286,0,10,BACKGROUND,IgE sensitization to Aspergillus fumigatus and...,0,0.000000,0,20,none,It is not clear whether these patients would b...,sentence 1 of 10 . previous: none next: It is ...
1,24290286,1,10,BACKGROUND,It is not clear whether these patients would b...,0,0.111111,1,13,IgE sensitization to Aspergillus fumigatus and...,We sought to determine whether a 3-month cours...,sentence 2 of 10 . previous: IgE sensitization...
2,24290286,2,10,OBJECTIVE,We sought to determine whether a 3-month cours...,1,0.222222,2,25,It is not clear whether these patients would b...,Asthmatic patients who were IgE sensitized to ...,sentence 3 of 10 . previous: It is not clear w...


In [7]:
for split, df in data.items():
    path = f'{DRIVE_DIR}/{DATA_VERSION}_{split}.csv'
    df.to_csv(path, index=False)
    print('Saved:', path)

for split in data:
    reloaded = pd.read_csv(f'{DRIVE_DIR}/{DATA_VERSION}_{split}.csv',
                           dtype={'abstract_id': str}, keep_default_na=False)
    assert reloaded.shape == data[split].shape and (reloaded.text == data[split].text).all(), f'{split}: mismatch'
print('✅ Files read back identically')

Saved: /content/drive/MyDrive/exsu_project/prepared_data/PubMed_20k_RCT_train.csv
Saved: /content/drive/MyDrive/exsu_project/prepared_data/PubMed_20k_RCT_dev.csv
Saved: /content/drive/MyDrive/exsu_project/prepared_data/PubMed_20k_RCT_test.csv
✅ Files read back identically
